# Friend AI Chat Clone — Colab Training

LoRA fine-tune Qwen-2.5-0.5B on a WhatsApp chat dump to mimic a friend's texting style.

**Time:** ~30-45 min (T4 GPU)
**Output:** GGUF model file you download and use locally.

## Step 1: Get scripts from GitHub + upload chat

Before running: create a GitHub token at https://github.com/settings/tokens/new?description=colab-ai-train&scopes=repo

Then add it in Colab: click the 🔑 key icon in left sidebar → Add new secret → name: `GITHUB_TOKEN`

In [ ]:
from google.colab import files, userdata
import os, shutil, glob, subprocess

# Clone scripts from GitHub
token = userdata.get('GITHUB_TOKEN')
if token:
    repo_url = f'https://{token}@github.com/zaheen4/ai-train.git'
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, '/tmp/ai-train'],
                   capture_output=True, text=True, check=True)
    for f in glob.glob('/tmp/ai-train/*.py') + glob.glob('/tmp/ai-train/*.sh'):
        shutil.copy2(f, '/content/')
        print(f'Script: {os.path.basename(f)}')
else:
    print('No GITHUB_TOKEN found. Upload scripts manually:')
    uploaded = files.upload()

# Upload chat file
print('\nUpload _chat.txt:')
uploaded = files.upload()
chat_file = list(uploaded.keys())[0]
print(f'Chat: {chat_file}')

## Step 2: Set your names

Enter your name and your friend's name exactly as they appear in the chat file.

In [ ]:
YOUR_NAME = 'Zaheen'  # <-- CHANGE THIS
FRIEND_NAME = 'Sucrose Jar'  # <-- CHANGE THIS
print(f'You: {YOUR_NAME}')
print(f'Friend: {FRIEND_NAME}')

## Step 3: Install Python + system dependencies

In [ ]:
!pip install -q torch transformers peft datasets accelerate scipy gguf trl sentencepiece 'torchao>=0.16.0' 2>&1 | tail -5
print('Python deps installed')

## Step 4: Copy scripts from Drive (already done in Step 1)

## Step 5: Start remote API server + Cloudflare Tunnel

Run this cell and wait for the tunnel URL to appear. Copy the URL and send it to the AI agent.

*Skip this if you're training manually.*

In [ ]:
import subprocess, threading, time, re, os, signal

# Download cloudflared
import urllib.request
urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '/usr/local/bin/cloudflared'
)
os.chmod('/usr/local/bin/cloudflared', 0o755)

# Launch the API server on port 8765
api_proc = subprocess.Popen(
    ['python3', 'colab_api_server.py'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(1)
print('API server started on port 8765')

# Launch cloudflared tunnel
log_file = '/tmp/cloudflared.log'
with open(log_file, 'w') as f:
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8765'],
        stdout=f, stderr=subprocess.STDOUT, text=True,
    )

# Wait for tunnel URL to appear
tunnel_url = None
for _ in range(30):
    time.sleep(1)
    with open(log_file, 'r') as f:
        content = f.read()
    m = re.search(r'https://[^\s]+\.trycloudflare\.com', content)
    if m:
        tunnel_url = m.group(0)
        break

print()
print('=' * 60)
print(f'  TUNNEL URL: {tunnel_url}')
print('=' * 60)
print()
print('COPY this URL and send it to the AI agent.')
print('Keep this Colab tab open — do NOT close it.')

---

## Manual mode

If you don't want to use the remote agent, run cells below manually.

### Step 6: Parse chat

In [ ]:
!python parse_chat.py "{chat_file}" "{YOUR_NAME}" "{FRIEND_NAME}"

### Step 7: Prepare training data

In [ ]:
pairs_file = chat_file.replace('.txt', '.pairs.json')
!python prepare_data.py "{pairs_file}" --model qwen --context 2
os.makedirs('working', exist_ok=True)
!cp train.jsonl working/ && cp valid.jsonl working/

### Step 8: Train LoRA

Set `NUM_EPOCHS` below — 1 is ~2.5h, 2 is ~5h.

In [ ]:
# ─── EDIT THIS ────────────────────────────────────
os.environ['NUM_EPOCHS'] = '1'  # 1 = ~2.5h, 2 = ~5h
# ───────────────────────────────────────────────────
os.environ['MODEL_SIZE'] = '0.5b'
os.environ['TRAIN_FILE'] = 'working/train.jsonl'
os.environ['VALID_FILE'] = 'working/valid.jsonl'
os.environ['WORK_DIR'] = os.path.abspath('working')
os.environ['OMP_NUM_THREADS'] = '4'
# Optional: set HF_TOKEN for faster downloads (get from https://huggingface.co/settings/tokens)
# os.environ['HF_TOKEN'] = 'hf_...'
!python train.py

### Step 9: Convert to GGUF

In [ ]:
import os
os.environ['MODEL_DIR'] = os.path.abspath('working/merged-model')
os.environ['WORK_DIR'] = os.path.abspath('working')
os.environ['MODEL_NAME'] = 'friend-model'
!python convert_to_gguf.py

### Step 10: Download the model

In [ ]:
from google.colab import files
gguf_files = glob.glob('working/*.gguf')
if gguf_files:
    for f in gguf_files:
        files.download(f)
else:
    from shutil import make_archive
    make_archive('merged-model', 'zip', 'working/merged-model')
    files.download('merged-model.zip')

## Done!

After downloading:

1. **If you got a .gguf file:** Quantize locally:
   ```bash
   llama-quantize friend-model-f16.gguf friend-model-Q4_K_M.gguf Q4_K_M
   ```

2. **Chat with your model:**
   ```bash
   bash chat.sh friend-model-Q4_K_M.gguf "{FRIEND_NAME}"
   ```